# CLIP Latent Reading: Light Across Discursive Descriptions

**FabLight research project** — Investigating the representation of light in 18th-century painting  
at the intersection of art history, history of science, and technology.

---

## Research question

> *How does CLIP organize images and textual descriptions related to light  
> across art-historical, technical, and poetic discourses?*

Rather than treating the model as a neutral retrieval tool, this notebook approaches CLIP  
as a **cultural structure** whose latent space can be read and interpreted.

---

## Pipeline overview

1. Load CLIP model and processor  
2. Encode images (paintings)  
3. Encode text prompts (3 discourse registers: art-historical / technical / poetic)  
4. Compute cosine similarity matrix  
5. Dimensionality reduction (PCA, UMAP)  
6. Semantic axis: project onto *art_historical − technical* direction  
7. Interpret and annotate first observations

## 0. Setup

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Src dir: {SRC_DIR}")

Project root: /Users/shannon/Desktop/workspace_msh/clip-latent-reading
Src dir: /Users/shannon/Desktop/workspace_msh/clip-latent-reading/src


In [3]:
from multimodal_interpretability_pilot.utils import (
    load_prompts, load_image_metadata, load_images_pil,
    encode_images, encode_texts,
    cosine_similarity_matrix, top_k_matches,
    category_mean_embeddings, compute_semantic_axis, project_onto_axis,
    run_pca, run_umap,
    plot_text_projection, plot_combined_projection,
    plot_similarity_heatmap, plot_semantic_axis,
    save_embeddings, load_embeddings,
    FIGURES_DIR, EMBEDDINGS_DIR, IMAGES_DIR,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Images dir:   {IMAGES_DIR}")

Project root: /Users/shannon/Desktop/workspace_msh/clip-latent-reading
Images dir:   /Users/shannon/Desktop/workspace_msh/clip-latent-reading/data/images


## 1. Load CLIP model

In [4]:
import torch
from transformers import CLIPModel, CLIPProcessor

MODEL_ID = "openai/clip-vit-base-patch32"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_ID} on {DEVICE}...")
clip_model     = CLIPModel.from_pretrained(MODEL_ID).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(MODEL_ID)
clip_model.eval()

print("Model loaded.")
d = clip_model.config.projection_dim
print(f"Embedding dimension: {d}")

/Users/shannon/Desktop/workspace_msh/clip-latent-reading/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading openai/clip-vit-base-patch32 on cpu...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 83508.40it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.
Embedding dimension: 512


## 2. Load data

In [7]:
prompts_df  = load_prompts()
metadata_df = load_image_metadata()

print(f"Prompts: {len(prompts_df)}")
print(prompts_df.groupby('category').size().to_string())
print()
print(f"Images: {len(metadata_df)}")
display(prompts_df)

Prompts: 12
category
art_historical    4
poetic            4
technical         4

Images: 4


,id,text,category,subcategory
0,t01,"A young woman holds a lamp, her face illuminat...",art_historical,candlelit_scene
1,t02,An elderly figure examines an object by lanter...,art_historical,chiaroscuro
2,t03,A solitary reader bent over a book in an inter...,art_historical,reading_scene
3,t04,A seated figure reads by the precise glow of a...,art_historical,enlightenment_interior
4,t05,An Argand lamp with a hollow circular wick and...,technical,lamp_description
5,t06,A tin oil lantern with punched ventilation hol...,technical,lamp_description
6,t07,A cross-section diagram of a lamp mechanism sh...,technical,mechanism
7,t08,Measurement of luminous intensity in candlepow...,technical,photometry
8,t09,Warm amber light diffusing through a dim room ...,poetic,atmosphere
9,t10,The intimate zone of visibility created by a s...,poetic,phenomenological


In [8]:
# Load PIL images
# ──────────────────────────────────────────────────────────────────
# IMPORTANT: rename your image files to match the 'filename' column
# in data/image_metadata.csv  (e.g. IMG1_dawe_foldsone_femme.jpg)
# Place them in data/images/
# ──────────────────────────────────────────────────────────────────

image_tuples = load_images_pil()   # list of (image_id, PIL.Image)
image_ids  = [t[0] for t in image_tuples]
pil_images = [t[1] for t in image_tuples]

print(f"Loaded {len(pil_images)} images: {image_ids}")

KeyError: 'filename'

## 3. Encode with CLIP

In [ ]:
# Encode images
print("Encoding images...")
image_embeddings = encode_images(clip_model, clip_processor, pil_images, device=DEVICE)
print(f"image_embeddings.shape = {image_embeddings.shape}")

# Encode texts
print("Encoding texts...")
texts = prompts_df["text"].tolist()
text_embeddings = encode_texts(clip_model, clip_processor, texts, device=DEVICE)
print(f"text_embeddings.shape  = {text_embeddings.shape}")

In [ ]:
# Save embeddings for later reuse (no need to re-encode)
save_embeddings({"image": image_embeddings, "text": text_embeddings})

## 4. Cosine similarity matrix

In [ ]:
# Image × Text similarity  (N_images × N_texts)
sim_image_text = cosine_similarity_matrix(image_embeddings, text_embeddings)
print(f"Similarity matrix shape: {sim_image_text.shape}")
print()

# Top-3 text matches per image
matches = top_k_matches(
    sim_image_text,
    query_labels  = image_ids,
    target_labels = prompts_df["id"].tolist(),
    k=5,
)
print("Top-5 text matches per image:")
display(matches)

In [ ]:
# Heatmap
fig, ax = plot_similarity_heatmap(
    sim_image_text,
    row_labels = image_ids,
    col_labels = prompts_df["id"].tolist(),
    save_path  = FIGURES_DIR / "heatmap_image_text.png",
    title      = "Image × Text cosine similarity — CLIP",
)
plt.show()

In [ ]:
# ── OBSERVATION CELL ──────────────────────────────────────────────
# Write your first notes here after reading the heatmap.
#
# Questions to ask:
#  - Which images score highest with art_historical prompts (t01–t04)?
#  - Are technical prompts (t05–t08) systematically lower for art images?
#  - Does the Argand lamp (IMG4) score differently than the candle scenes?
#  - Do poetic prompts (t09–t12) score close to or distant from art_historical ones?
#
# NOTES:
# ...
# ──────────────────────────────────────────────────────────────────

## 5. Text–text similarity (intra-discourse structure)

How similar are the three discourse registers to each other in CLIP's embedding space?

In [ ]:
# Text × Text similarity
sim_text_text = cosine_similarity_matrix(text_embeddings, text_embeddings)

fig, ax = plot_similarity_heatmap(
    sim_text_text,
    row_labels = prompts_df["id"].tolist(),
    col_labels = prompts_df["id"].tolist(),
    save_path  = FIGURES_DIR / "heatmap_text_text.png",
    title      = "Text × Text cosine similarity — CLIP (categories t01–t12)",
)
plt.show()

# Category-level averages
categories = prompts_df["category"].tolist()
unique_cats = ["art_historical", "technical", "poetic"]
cat_indices = {c: [i for i, cat in enumerate(categories) if cat == c] for c in unique_cats}

print("\nMean within-category and cross-category similarities:")
for ca in unique_cats:
    for cb in unique_cats:
        idxA = cat_indices[ca]
        idxB = cat_indices[cb]
        block = sim_text_text[np.ix_(idxA, idxB)]
        if ca == cb:
            # Exclude diagonal (self-similarity = 1.0)
            mask = ~np.eye(len(idxA), dtype=bool)
            mean_sim = block[mask].mean()
        else:
            mean_sim = block.mean()
        print(f"  {ca[:10]:12} × {cb[:10]:12}  →  {mean_sim:.4f}")

## 6. Dimensionality reduction — PCA

In [ ]:
from sklearn.decomposition import PCA

# 6a. PCA on text embeddings alone
text_pca_coords, text_explained = run_pca(text_embeddings)
print(f"PCA text: explained variance = {text_explained}")

fig, ax = plot_text_projection(
    text_pca_coords,
    prompts_df,
    method_name  = "PCA",
    explained_var= text_explained,
    save_path    = FIGURES_DIR / "pca_text_embeddings.png",
)
plt.show()

In [ ]:
# 6b. Joint PCA on image + text embeddings
all_embeddings = np.vstack([text_embeddings, image_embeddings])
joint_coords, joint_explained = run_pca(all_embeddings)
print(f"PCA joint: explained variance = {joint_explained}")

text_joint_coords  = joint_coords[:len(text_embeddings)]
image_joint_coords = joint_coords[len(text_embeddings):]

fig, ax = plot_combined_projection(
    text_joint_coords,
    image_joint_coords,
    prompts_df,
    image_ids,
    method_name  = "PCA",
    explained_var= joint_explained,
    save_path    = FIGURES_DIR / "pca_joint_embeddings.png",
)
plt.show()

In [ ]:
# ── OPTIONAL: UMAP ────────────────────────────────────────────────
# Only run if umap-learn is installed: pip install umap-learn
# Comment out if it causes trouble today

try:
    umap_text_coords = run_umap(text_embeddings, n_neighbors=5, min_dist=0.2)
    fig, ax = plot_text_projection(
        umap_text_coords,
        prompts_df,
        method_name = "UMAP",
        save_path   = FIGURES_DIR / "umap_text_embeddings.png",
    )
    plt.show()
except ImportError:
    print("umap-learn not installed — skipping UMAP. Run: pip install umap-learn")

## 7. Semantic axis: art-historical vs technical

We construct a direction vector in embedding space from the mean of *technical* prompts  
to the mean of *art-historical* prompts. Then we project all embeddings (images + texts)  
onto this direction to see where they fall.

**Hypothesis:** paintings should cluster toward the art-historical end;  
technical lamp descriptions should cluster at the technical end;  
poetic prompts may occupy an intermediate position.

In [ ]:
# Compute per-category mean embeddings
categories    = prompts_df["category"].tolist()
cat_means     = category_mean_embeddings(text_embeddings, categories)

print("Category mean embedding norms:")
for cat, vec in cat_means.items():
    print(f"  {cat:20s}  norm = {np.linalg.norm(vec):.4f}")

# Semantic axis: art_historical − technical
axis = compute_semantic_axis(cat_means, cat_a="art_historical", cat_b="technical")
print(f"\nAxis norm (should be 1.0): {np.linalg.norm(axis):.4f}")

# Project
image_scores = project_onto_axis(image_embeddings, axis)
text_scores  = project_onto_axis(text_embeddings, axis)

print("\nImage projection scores (positive → art-historical, negative → technical):")
for img_id, score in zip(image_ids, image_scores):
    print(f"  {img_id}: {score:+.4f}")

print("\nText projection scores by category:")
for i, row in prompts_df.iterrows():
    print(f"  {row['id']} [{row['category']:15s}]: {text_scores[i]:+.4f}")

In [ ]:
fig, ax = plot_semantic_axis(
    image_scores,
    image_ids,
    text_scores  = text_scores,
    prompts_df   = prompts_df,
    save_path    = FIGURES_DIR / "semantic_axis_art_vs_technical.png",
)
plt.show()

In [ ]:
# Cross-category distance in embedding space
from itertools import combinations
print("Distances between category mean embeddings:")
unique_cats = list(cat_means.keys())
for ca, cb in combinations(unique_cats, 2):
    dist = np.linalg.norm(cat_means[ca] - cat_means[cb])
    cos  = float(cat_means[ca] @ cat_means[cb] / 
                 (np.linalg.norm(cat_means[ca]) * np.linalg.norm(cat_means[cb])))
    print(f"  {ca:20s} ↔ {cb:20s}  L2={dist:.4f}  cos={cos:.4f}")

## 8. First observations & interpretation

**Use this cell to write your interpretive notes — these become the README draft.**

---

### 8.1 Similarity matrix observations

_(fill in after running cell 4)_

- ...
- ...

### 8.2 PCA structure

_(fill in after running cell 6)_

- ...
- ...

### 8.3 Semantic axis

_(fill in after running cell 7)_

- Do the paintings cluster toward the art-historical end?
- Where does the Argand lamp (IMG4/Kersting) fall compared to candle scenes?
- Are poetic and art-historical prompts close to each other in this space?
- ...

### 8.4 Research implications

- What does this suggest about how CLIP handles cross-discursive concepts?
- Is CLIP able to distinguish 'light as artistic device' from 'light as technical object'?
- ...

## 9. Export summary table

Useful for the README and for sharing results.

In [ ]:
# Per-image: top match per discourse category
results_rows = []

for img_idx, img_id in enumerate(image_ids):
    row = {"image_id": img_id}
    for cat in ["art_historical", "technical", "poetic"]:
        cat_mask = prompts_df["category"] == cat
        cat_indices_list = prompts_df[cat_mask].index.tolist()
        cat_sims = sim_image_text[img_idx, cat_mask.values]
        best_idx = np.argmax(cat_sims)
        best_id  = prompts_df[cat_mask]["id"].iloc[best_idx]
        row[f"best_{cat}"] = best_id
        row[f"sim_{cat}"]  = round(float(cat_sims[best_idx]), 4)
    row["semantic_axis_score"] = round(float(image_scores[img_idx]), 4)
    results_rows.append(row)

results_df = pd.DataFrame(results_rows)
results_df.to_csv(EMBEDDINGS_DIR / "results_summary.csv", index=False)
print("Saved results_summary.csv")
display(results_df)

---

## Appendix — extending the experiment

When you come back to this project, next steps might include:

1. **Larger image corpus**: add scientific illustration images (lamp diagrams, photometry apparatus)
2. **Prompt expansion**: add French-language prompts to test multilingual CLIP behavior
3. **Multiple discourse directions**: not just art_historical vs technical, but also  
   art_historical vs poetic, or a three-way ternary plot
4. **Attention map probing**: use GradCAM or attention rollout to see *where* in the image  
   CLIP attends when matching different discourse categories
5. **Controlled prompt variations**: systematically vary one word (e.g. 'candle' / 'lamp' / 'lantern')  
   and trace the movement in embedding space
6. **Comparison with CLIP variants**: ViT-L/14, OpenCLIP, multilingual CLIP